# 🧬 16. Feature Engineering: RGB + AVG + NTSC Feature Fusion Ekstraksi

Notebook ini menggabungkan (Feature Fusion) representasi fitur tingkat tinggi dari **3 Model CNN terlatih**:
1. **RGB CNN Features** (512 dimensi dari `cifar10_artifacts/rgb`)
2. **Grayscale AVG CNN Features** (512 dimensi dari `cifar10_artifacts/grayscale_avg`)
3. **Grayscale NTSC CNN Features** (512 dimensi dari `cifar10_artifacts/grayscale_ntsc`)

### 📌 Aturan Feature Fusion:
- **TIDAK MELATIH CNN BARU**: Murni memanfaatkan feature vector yang telah diekstrak oleh ketiga CNN spesialis sebelumnya.
- **Combined Feature Vector**: `512 + 512 + 512 = 1.536 dimensi`.
- **Target Artifact**: `cifar10_artifacts/rgb_avg_ntsc/train_features.npz` dan `test_features.npz`.


In [1]:
import os
import sys
import numpy as np


ARTIFACT_DIR = 'cifar10_artifacts/rgb_avg_ntsc'
os.makedirs(ARTIFACT_DIR, exist_ok=True)
print(f"Artifact directory: {ARTIFACT_DIR}")

# Lokasi fitur dari ketiga model CNN
rgb_dir  = 'cifar10_artifacts/rgb'
avg_dir  = 'cifar10_artifacts/grayscale_avg'
ntsc_dir = 'cifar10_artifacts/grayscale_ntsc'

print("Memuat vektor fitur training dan testing dari 3 workflow sebelumnya...")
rgb_train  = np.load(os.path.join(rgb_dir, 'train_features.npz'))
avg_train  = np.load(os.path.join(avg_dir, 'train_features.npz'))
ntsc_train = np.load(os.path.join(ntsc_dir, 'train_features.npz'))

rgb_test   = np.load(os.path.join(rgb_dir, 'test_features.npz'))
avg_test   = np.load(os.path.join(avg_dir, 'test_features.npz'))
ntsc_test  = np.load(os.path.join(ntsc_dir, 'test_features.npz'))

y_train = rgb_train['y_train']
y_test  = rgb_test['y_test']


Artifact directory: cifar10_artifacts/rgb_avg_ntsc
Memuat vektor fitur training dan testing dari 3 workflow sebelumnya...


## 📥 1. Feature Fusion (Concatenation) Menjadi Vektor 1.536-Dimensi


In [2]:
print("Menggabungkan fitur training (RGB + AVG + NTSC)...")
X_train_fused = np.concatenate([
    rgb_train['X_train_features'],
    avg_train['X_train_features'],
    ntsc_train['X_train_features']
], axis=1)

print("Menggabungkan fitur testing (RGB + AVG + NTSC)...")
X_test_fused = np.concatenate([
    rgb_test['X_test_features'],
    avg_test['X_test_features'],
    ntsc_test['X_test_features']
], axis=1)

print()
print(f"[SELESAI] X_train_fused shape : {X_train_fused.shape} (50.000 sampel, 1.536 dimensi)")
print(f"[SELESAI] X_test_fused shape  : {X_test_fused.shape} (10.000 sampel, 1.536 dimensi)")


Menggabungkan fitur training (RGB + AVG + NTSC)...


Menggabungkan fitur testing (RGB + AVG + NTSC)...



[SELESAI] X_train_fused shape : (50000, 1536) (50.000 sampel, 1.536 dimensi)
[SELESAI] X_test_fused shape  : (10000, 1536) (10.000 sampel, 1.536 dimensi)


## 💾 2. Menyimpan Vektor Fitur Gabungan ke Disk (.npz)


In [3]:
train_feat_path = os.path.join(ARTIFACT_DIR, 'train_features.npz')
test_feat_path  = os.path.join(ARTIFACT_DIR, 'test_features.npz')

# Pertahankan y_pred_svm yang terkalibrasi jika sudah ada
existing_pred = None
if os.path.exists(test_feat_path):
    try:
        prev_data = np.load(test_feat_path)
        if 'y_pred_svm' in prev_data:
            existing_pred = prev_data['y_pred_svm']
    except Exception:
        pass

np.savez_compressed(train_feat_path, X_train_features=X_train_fused, y_train=y_train)
if existing_pred is not None:
    np.savez_compressed(test_feat_path, X_test_features=X_test_fused, y_test=y_test, y_pred_svm=existing_pred)
else:
    np.savez_compressed(test_feat_path, X_test_features=X_test_fused, y_test=y_test)

print(f"[BERHASIL] Fitur gabungan training tersimpan: {train_feat_path}")
print(f"[BERHASIL] Fitur gabungan testing tersimpan : {test_feat_path}")


[BERHASIL] Fitur gabungan training tersimpan: cifar10_artifacts/rgb_avg_ntsc\train_features.npz
[BERHASIL] Fitur gabungan testing tersimpan : cifar10_artifacts/rgb_avg_ntsc\test_features.npz
